In [2]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

In [3]:
!pip install -q \
requests \
python-dotenv \
tqdm \
groq \
SPARQLWrapper \
beautifulsoup4 \
networkx \
pyvis \
scikit-learn \
playwright \
nest_asyncio \

# khusus playwright
!pip install playwright
!playwright install chromium

In [ ]:
# # Khusus colab untuk support playwright/selenium
# !apt-get update
# !apt-get install -y \
# libatk1.0-0 \
# libatk-bridge2.0-0 \
# libgtk-3-0 \
# libxcomposite1 \
# libxdamage1 \
# libxfixes3 \
# libxrandr2 \
# libgbm1 \
# libpango-1.0-0 \
# libasound2 \
# libxkbcommon0

# !playwright install chromium

In [4]:
# ── Standard Library ──────────────────────────────────────────────────────────
import os
import re
import json
import math
import time
import random
import asyncio
import threading

from collections import defaultdict
from datetime import datetime, timezone
from functools import lru_cache

# URL & HTTP Utilities
from urllib.parse import (
    urlparse,
    urljoin,
    urlencode,
    parse_qs,
    quote,
    unquote,
)
from urllib.request import urlopen, Request
from urllib.error import HTTPError, URLError

from email.utils import parsedate_to_datetime

# ── Data & Numerics ───────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualization ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
from pyvis.network import Network

# ── Web & API ─────────────────────────────────────────────────────────────────
import requests
from bs4 import BeautifulSoup
from groq import Groq
from SPARQLWrapper import SPARQLWrapper, JSON
from tqdm import tqdm

# ── Environment Variables ─────────────────────────────────────────────────────
from dotenv import load_dotenv
from kaggle_secrets import UserSecretsClient

# ── Playwright (Browser Automation) ───────────────────────────────────────────
from playwright.sync_api import sync_playwright
from playwright.async_api import async_playwright

# ── Async Fix for Notebook ────────────────────────────────────────────────────
import nest_asyncio
nest_asyncio.apply()

# ── Graph Analysis ────────────────────────────────────────────────────────────
import networkx as nx
from networkx.algorithms.community import girvan_newman

# ── NLP & Embedding ───────────────────────────────────────────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer

# ── Machine Learning ──────────────────────────────────────────────────────────
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight

# ── Date Parsing ──────────────────────────────────────────────────────────────
from dateutil import parser

# Get Data

In [5]:
import re
from SPARQLWrapper import SPARQLWrapper, JSON

WIKIDATA_ENDPOINT = "https://query.wikidata.org/sparql"
WIKIDATA_ENTITY_PFX = "http://www.wikidata.org/entity/"

_formatter_cache: dict[str, str] = {}

# This function constructs a full URL using the Wikidata formatter URL pattern (P1630) by replacing the '$1' placeholder with the actual external ID.
# Case: in stated in references some of it actually a website but in query we just get the incomplete URL
def get_external_url(property_id: str, value_id: str, value_uri: str) -> str:
    if re.match(r"^Q\d+$", value_id):
        return ""
    if value_uri.startswith(("http://", "https://")):
        return value_uri
    formatter = _formatter_cache.get(property_id, "")
    if formatter and "$1" in formatter:
        return formatter.replace("$1", value_id)
    return ""

In [6]:
# ─────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────

# This function splits a SPARQL grouped string (separated by "||") into a standard Python list.
def _split_field(binding: dict, key: str) -> list[str]:
    raw = binding.get(key, {}).get("value", "")
    return [v for v in raw.split("||") if v] if raw else []

# This function parses the raw reference bundle from Wikidata into a structured dictionary of source metadata.
def _parse_sources(ref_bundles_raw: str, prop_id: str) -> list[dict]:
    sources:   list[dict] = []
    seen_refs: set[str]   = set()

    if not ref_bundles_raw:
        return sources

    for bundle in ref_bundles_raw.split("||||"):
        parts = bundle.split("~")
        parts += [""] * (13 - len(parts))

        ref_id = parts[0].strip()
        if not ref_id or ref_id in seen_refs:
            continue
        seen_refs.add(ref_id)

        imported_from       = parts[11].strip()
        imported_from_label = parts[12].strip()

        if imported_from:
            ref_type = "imported_from_wikimedia"
        elif parts[2].strip():
            ref_type = "stated_in"
        elif parts[1].strip():
            ref_type = "url"
        else:
            ref_type = "unknown"

        stated_in_raw  = parts[2].strip()
        stated_in_id   = stated_in_raw.split("/")[-1] if stated_in_raw else ""
        stated_in_site = parts[4].strip()

        if not stated_in_site and stated_in_id:
            stated_in_site = get_external_url(prop_id, stated_in_id, stated_in_raw)

        sources.append({
            "ref_id":              ref_id,
            "ref_type":            ref_type,
            "url":                 parts[1].strip(),
            "stated_in_id":        stated_in_id,
            "stated_in_label":     parts[3].strip(),
            "stated_in_website":   stated_in_site,
            "publisher_label":     parts[5].strip(),
            "publisher_website":   parts[6].strip(),
            "retrieved":           parts[7].strip(),
            "published":           parts[8].strip(),
            "archive_date":        parts[9].strip(),
            "archive_url":         parts[10].strip(),
            "imported_from":       imported_from.split("/")[-1] if imported_from else "",
            "imported_from_label": imported_from_label,
        })

    return sources

In [7]:
WIKIDATA_API = "https://www.wikidata.org/w/api.php"

_property_cache: dict[str, str] = {}

# This function resolves a natural language property label (e.g., "head of government") into its corresponding Wikidata Property ID (e.g., "P6").
def find_property_id(label: str) -> str:
    normalized = label.strip().lower()
    if normalized in _property_cache:
        return _property_cache[normalized]

    params = {
        "action": "wbsearchentities",
        "search": label,
        "language": "en",
        "type": "property",
        "format": "json",
        "limit": 10,
    }
    resp = _session.get(WIKIDATA_API, params=params, timeout=20)
    resp.raise_for_status()
    data = resp.json()

    if not data.get("search"):
        raise ValueError(f"Property '{label}' not found")

    pid = next(
        (r["id"].upper() for r in data["search"]
         if r.get("label", "").lower() == normalized),
        data["search"][0]["id"].upper(),
    )
    _property_cache[normalized] = pid
    return pid

In [9]:
# ───────────────────────────────────────────
# Helper Filter by Property (head of government, etc)
# ─────────────────────────────────────────────

# This function standardizes a list of mixed property formats (IDs or labels) into a clean list of Wikidata Property IDs.
def resolve_properties(properties: list[str]) -> list[str]:
    resolved = []
    seen = set()

    for raw in properties:
        prop = raw.strip()

        try:
            if re.fullmatch(r"P\d+", prop, re.IGNORECASE):
                pid = prop.upper()

            elif m := re.search(r"(P\d+)", prop, re.IGNORECASE):
                pid = m.group(1).upper()

            else:
                pid = find_property_id(prop)

            if pid and pid not in seen:
                seen.add(pid)
                resolved.append(pid)

        except Exception as exc:
            print(f"⚠️ Skip property '{prop}': {exc}")

    return resolved

In [10]:
# Helper
def clean_wikidata_time_claim(value):

    if not isinstance(value, str):
        return value

    if value.endswith("T00:00:00Z"):
        return value[:10]

    return value

In [11]:
def sparql_query_with_retry(query: str, max_retries: int = 5) -> dict:
    params  = {"query": query, "format": "json"}
    headers = {"Accept": "application/sparql-results+json"}

    for attempt in range(1, max_retries + 1):
        _adaptive_sleep()
        try:
            resp = _session.get(
                WIKIDATA_ENDPOINT,
                params=params,
                headers=headers,
                timeout=30,
            )

            if resp.status_code == 429:
                retry_after = int(resp.headers.get("Retry-After", 0))
                _on_rate_limited(retry_after)
                print(f"429 rate limited (attempt {attempt}/{max_retries}), waiting {_min_interval:.1f}s")
                time.sleep(_min_interval)
                continue

            if resp.status_code in {500, 502, 503, 504}:
                wait = 2 ** (attempt - 1)
                print(f"HTTP {resp.status_code} (attempt {attempt}/{max_retries}), retrying in {wait}s")
                time.sleep(wait)
                continue

            resp.raise_for_status()
            _on_success()
            return resp.json()

        except requests.exceptions.Timeout:
            wait = 2 ** (attempt - 1)
            print(f"Timeout (attempt {attempt}/{max_retries}), retrying in {wait}s")
            time.sleep(wait)

        except requests.exceptions.ConnectionError as exc:
            wait = 2 ** (attempt - 1)
            print(f"Connection error: {exc} (attempt {attempt}/{max_retries}), retrying in {wait}s")
            time.sleep(wait)

    raise RuntimeError(f"SPARQL query failed after {max_retries} retries")

In [12]:
USER_AGENT = "WikidataClaimsTrustworthinessTimelinessAnalyzer"

# ---------------------------------------------------------------------------
# Shared HTTP session — connection pool + User-Agent for all outbound requests
# ---------------------------------------------------------------------------

_session = requests.Session()
_session.headers.update({"User-Agent": USER_AGENT})

# ---------------------------------------------------------------------------
# Caches
# ---------------------------------------------------------------------------

_formatter_cache: dict[str, str] = {}   # prop_id  -> formatter URL (P1630)
_property_cache:  dict[str, str] = {}   # label    -> property ID

# ---------------------------------------------------------------------------
# Adaptive rate-limit state
# ---------------------------------------------------------------------------

_rate_lock:         threading.Lock = threading.Lock()
_last_request_time: float          = 0.0
_min_interval:      float          = 0.6    # seconds; grows on 429, shrinks on success
_MAX_INTERVAL:      float          = 30.0
_BACKOFF_FACTOR:    float          = 2.0
_RECOVERY_FACTOR:   float          = 0.9

# ---------------------------------------------------------------------------
# Rate-limit helpers
# ---------------------------------------------------------------------------

def _adaptive_sleep() -> None:
    global _last_request_time, _min_interval
    with _rate_lock:
        elapsed = time.monotonic() - _last_request_time
        wait    = _min_interval - elapsed
        if wait > 0:
            time.sleep(wait)
        _last_request_time = time.monotonic()


def _on_success() -> None:
    global _min_interval
    with _rate_lock:
        _min_interval = max(0.6, _min_interval * _RECOVERY_FACTOR)


def _on_rate_limited(retry_after: int = 0) -> None:
    global _min_interval
    with _rate_lock:
        _min_interval = min(
            _MAX_INTERVAL,
            max(_min_interval * _BACKOFF_FACTOR, float(retry_after))
        )

In [14]:
def get_entity_claims(
    entity_qid: str = "Q252",
    limit: int | None = None,
    properties: list[str] | None = None,
) -> list[dict]:
    resolved_pids: list[str] = []
    if properties:
        resolved_pids = resolve_properties(properties)

    values_clause = (
        "VALUES ?property { " + " ".join(f"wd:{p}" for p in resolved_pids) + " }"
        if resolved_pids else ""
    )
    limit_clause = f"LIMIT {limit}" if limit is not None else ""

    query = f"""
    PREFIX wd:       <http://www.wikidata.org/entity/>
    PREFIX p:        <http://www.wikidata.org/prop/>
    PREFIX ps:       <http://www.wikidata.org/prop/statement/>
    PREFIX pq:       <http://www.wikidata.org/prop/qualifier/>
    PREFIX pr:       <http://www.wikidata.org/prop/reference/>
    PREFIX prov:     <http://www.w3.org/ns/prov#>
    PREFIX wikibase: <http://wikiba.se/ontology#>
    PREFIX bd:       <http://www.bigdata.com/rdf#>
    PREFIX wdt:      <http://www.wikidata.org/prop/direct/>
    PREFIX xsd:      <http://www.w3.org/2001/XMLSchema#>

    SELECT ?entity ?entityLabel
           ?property ?propertyLabel
           ?value ?valueLabel
           ?valueType
           ?formatterUrl

           (GROUP_CONCAT(DISTINCT ?pointInTime; separator="||") AS ?pointInTimes)
           (GROUP_CONCAT(DISTINCT ?startTime;   separator="||") AS ?startTimes)
           (GROUP_CONCAT(DISTINCT ?endTime;     separator="||") AS ?endTimes)

           (COUNT(DISTINCT ?ref) AS ?refCount)
           (GROUP_CONCAT(DISTINCT ?refBundle;   separator="||||") AS ?refBundles)

    WHERE {{
      BIND(wd:{entity_qid} AS ?entity)

      {values_clause}

      ?entity ?p ?statement .
      FILTER(STRSTARTS(STR(?p), STR(p:)))

      ?property wikibase:claim ?p .
      ?property wikibase:statementProperty ?ps .
      ?statement ?ps ?value .

      BIND(
        IF(STRSTARTS(STR(?value), "{WIKIDATA_ENTITY_PFX}"), "wikibase-entityid",
          IF(isIRI(?value), "iri",
            IF(DATATYPE(?value) = xsd:dateTime, "time",
              IF(DATATYPE(?value) = xsd:decimal || DATATYPE(?value) = xsd:integer,
                 "quantity",
                 "string"
              )
            )
          )
        ) AS ?valueType
      )

      OPTIONAL {{ ?property wdt:P1630 ?formatterUrl . }}

      OPTIONAL {{ ?statement pq:P585 ?pointInTime . }}
      OPTIONAL {{ ?statement pq:P580 ?startTime . }}
      OPTIONAL {{ ?statement pq:P582 ?endTime . }}

      OPTIONAL {{
        ?statement prov:wasDerivedFrom ?ref .

        OPTIONAL {{ ?ref pr:P854  ?refUrl . }}
        OPTIONAL {{ ?ref pr:P813  ?retrieved . }}
        OPTIONAL {{ ?ref pr:P2960 ?archiveDate . }}
        OPTIONAL {{ ?ref pr:P1065 ?archiveUrl . }}
        OPTIONAL {{ ?ref pr:P577  ?published_ref . }}

        OPTIONAL {{
          ?ref pr:P143 ?importedFrom .
          OPTIONAL {{
            ?importedFrom rdfs:label ?importedFromLabel .
            FILTER(LANG(?importedFromLabel) = "en")
          }}
        }}

        OPTIONAL {{
          ?ref pr:P248 ?statedIn .
          OPTIONAL {{
            ?statedIn rdfs:label ?statedInLabel .
            FILTER(LANG(?statedInLabel) = "en")
          }}
          OPTIONAL {{ ?statedIn wdt:P856 ?statedInWebsite . }}
          OPTIONAL {{
            ?statedIn wdt:P123 ?publisher .
            OPTIONAL {{
              ?publisher rdfs:label ?publisherLabel .
              FILTER(LANG(?publisherLabel) = "en")
            }}
            OPTIONAL {{ ?publisher wdt:P856 ?publisherWebsite . }}
          }}
          OPTIONAL {{ ?statedIn wdt:P577 ?published_item . }}
        }}

        BIND(COALESCE(?published_ref, ?published_item) AS ?published)
        BIND(STR(?ref) AS ?refId)

        BIND(CONCAT(
          COALESCE(STR(?refId),             ""), "~",
          COALESCE(STR(?refUrl),            ""), "~",
          COALESCE(STR(?statedIn),          ""), "~",
          COALESCE(STR(?statedInLabel),     ""), "~",
          COALESCE(STR(?statedInWebsite),   ""), "~",
          COALESCE(STR(?publisherLabel),    ""), "~",
          COALESCE(STR(?publisherWebsite),  ""), "~",
          COALESCE(STR(?retrieved),         ""), "~",
          COALESCE(STR(?published),         ""), "~",
          COALESCE(STR(?archiveDate),       ""), "~",
          COALESCE(STR(?archiveUrl),        ""), "~",
          COALESCE(STR(?importedFrom),      ""), "~",
          COALESCE(STR(?importedFromLabel), "")
        ) AS ?refBundle)
      }}

      SERVICE wikibase:label {{
        bd:serviceParam wikibase:language "en".
      }}
    }}
    GROUP BY ?entity ?entityLabel
             ?property ?propertyLabel
             ?value ?valueLabel
             ?valueType ?formatterUrl
    {limit_clause}
    """

    results = sparql_query_with_retry(query)

    claims: list[dict] = []
    formatter_updates: dict[str, str] = {}

    for r in results["results"]["bindings"]:
        prop_id    = r["property"]["value"].split("/")[-1]
        prop_label = r.get("propertyLabel", {}).get("value", prop_id)

        value_uri   = r["value"]["value"]
        value_label = clean_wikidata_time_claim(
            r.get("valueLabel", {}).get("value", value_uri)
        )
        value_id = (
            value_uri.split("/")[-1]
            if value_uri.startswith(WIKIDATA_ENTITY_PFX)
            else value_uri
        )
        value_type   = r.get("valueType",   {}).get("value", "string")
        entity_label = r.get("entityLabel", {}).get("value", entity_qid)
        ref_count    = int(r.get("refCount", {}).get("value", 0))

        fmt_url = r.get("formatterUrl", {}).get("value", "")
        if fmt_url and prop_id not in _formatter_cache:
            formatter_updates[prop_id] = fmt_url

        external_url = get_external_url(prop_id, value_id, value_uri)
        sources      = _parse_sources(r.get("refBundles", {}).get("value", ""), prop_id)

        claims.append({
            "entity":         entity_qid,
            "entity_label":   entity_label,
            "property_id":    prop_id,
            "property_label": prop_label,
            "value":          value_label,
            "value_id":       value_id,
            "value_type":     value_type,
            "value_url":      external_url,
            "claim":          f"{entity_label} — {prop_label}: {value_label}",
            "ref_count":      ref_count,
            "sources":        sources,
            "qualifiers": {
                "point_in_time": _split_field(r, "pointInTimes"),
                "start_time":    _split_field(r, "startTimes"),
                "end_time":      _split_field(r, "endTimes"),
            },
        })

    _formatter_cache.update(formatter_updates)
    return claims

In [15]:
# Get Data
entities = ["Q252", "Q2107268", "Q1065", "Q95", "Q4261459", "Q3630", "Q6088540", "Q28752050", "Q615", "Q25188"]
# Indonesia, Prabowo, United Nations, Google, Gerindra, Jakarta, Istana Negara, Pemilu 2019, Messi, Film Inception
# entities = ["Q252"]

# limit = 100

print("🔍 Fetching Wikidata claims...")
claims = []
for ent in entities:
    # claims.extend(get_entity_claims(entity_qid=ent, limit=None))
    claims.extend(get_entity_claims(entity_qid=ent, properties=["main regulatory text"]))
    time.sleep(5) # avoid too many request

print(f"📦 Total claims fetched: {len(claims)}")

if not claims:
    print("❌ No claims found. Stop.")
    exit()

🔍 Fetching Wikidata claims...
📦 Total claims fetched: 1


# Validate SPO

In [ ]:
# This function safely escapes special string characters to prevent SPARQL injection or query malformations.
def escape_sparql_string(s):
    return (
        s.replace("\\", "\\\\")
         .replace('"', '\\"')
         .replace("\n", "\\n")
         .replace("\r", "\\r")
    )

In [ ]:
WIKIDATA_ENDPOINT = "https://query.wikidata.org/sparql"

# A helper function to escape special characters in SPARQL string literals and match sparql format -> Q252 menjadi wd:Q252
def format_value(value, value_type: str = None) -> str:
    if value is None:
        raise ValueError("value is None")
    if isinstance(value, str):
        value = value.strip()

    # url eksternal pada value
    if value_type == "iri":
        return f"<{value}>"

    # If value_type not string/external-id
    # to detect if value is IRI/URI
    # decide write it as <http://example.com> or string literal
    if isinstance(value, str) and re.match(r'^[a-zA-Z][a-zA-Z0-9+\-.]*:', value):
        if value_type not in ("string", "monolingualtext", "external-id"):
            return f"<{value}>"
        # value_type=string/external-id → fall through to quoted literal

    # Entity reference
    if value_type == "wikibase-entityid":
        return f"wd:{value}"

    # Explicit literal types 
    # value_type="string" formatted as literal
    # even the python type is int
    if value_type in ("string", "monolingualtext", "quantity", "external-id"):
        return f'"{escape_sparql_string(str(value))}"'

    # Native Python numbers (if value_type unknown)
    if isinstance(value, int):
        return f'"{value}"'
    if isinstance(value, float):
        return f'"{int(value)}"' if value == int(value) else f'"{value}"'

    # Fallback
    return f'"{escape_sparql_string(str(value))}"'

In [ ]:
# Remove extra whitespace and ensure consistent string representation for literals
def normalize_value(value) -> str | None:
    """
    Normalize a SPARQL result value to a plain string for dict-key lookup.

    Handles:
      - Entity URIs       : "http://www.wikidata.org/entity/Q42" → "Q42"
      - Angle-bracket IRI : "<https://example.com>"              → "https://example.com"
      - Plain IRI (uri)   : "https://example.com"                → "https://example.com"
      - Typed literals    : '"123"^^xsd:integer'                 → "123"
      - Quantity with sign: '"+1234"^^xsd:decimal'               → "1234"
      - Unit quantities   : '"1234"^^<http://units/...>'         → "1234"
      - Language tags     : '"hello"@en'                         → "hello"
      - Escaped quotes    : 'או\"מ'                              → 'או"מ'
    """
    if value is None:
        return None

    if isinstance(value, str) and "wikidata.org/entity/" in value:
        return value.split("/")[-1]

    if isinstance(value, str):
        stripped = value.strip()

        # Angle-bracket IRI → strip brackets
        if stripped.startswith("<") and stripped.endswith(">"):
            return stripped[1:-1]

        # Plain IRI from SPARQL result (type=uri) — return as-is
        # This handles url-type properties stored as RDF IRIs in Wikidata
        if re.match(r'^[a-zA-Z][a-zA-Z0-9+\-.]*:', stripped):
            return stripped

        # Strip datatype annotation including IRI-form units:
        # "1234"^^xsd:decimal       → "1234"
        # "1234"^^<http://unit/...> → "1234"
        stripped = re.sub(r'\^\^(<[^>]+>|[^\s"]+)', "", stripped)
        # Strip language tag: "hello"@en → "hello"
        stripped = re.sub(r'@[a-zA-Z\-]+$', "", stripped.strip())
        # Strip surrounding quotes then unescape inner quotes
        stripped = stripped.strip().strip('"').strip("'")
        stripped = stripped.replace('\\"', '"').replace("\\'", "'")
        # Strip leading '+' from signed quantities: "+1234" → "1234"
        if stripped.startswith("+"):
            stripped = stripped[1:]

        return stripped

    return str(value)

In [ ]:
def _sparql_val_to_lookup_key(sparql_val: str) -> str:
    """
    Convert a SPARQL term string (as produced by format_value) to the plain
    string used as the lookup key in result_map.

    Examples:
      "wd:Q42"            → "Q42"
      "<https://foo.com>" → "https://foo.com"
      '"hello"'           → "hello"
    """
    if sparql_val.startswith("wd:"):
        return sparql_val[3:]                # entity reference → bare QID
    return normalize_value(sparql_val)       # IRI or literal → plain string

In [ ]:
def validate_triples_batch(claims: list[dict], batch_size: int = 25, delay: float = 2.0) -> list[dict]:
    """
    Validate (entity, property, value) triples against Wikidata SPARQL.

    Args:
        claims     : Output of get_entity_claims(). Each item must contain
                     'entity', 'property_id', 'value_id', and optionally
                     'value_type' (from the Wikidata API).
        batch_size : Unique (entity, value) pairs per request.
        delay      : Sleep in seconds between requests. Wikidata anonymous
                     limit is 1 request per 5 seconds.

    Returns:
        The same list with these fields added to every item:
            triple_exists      (bool)
            triple_rank        (str | None)
            ref_count          (int)
            validation_skipped (bool)
            skip_reason        (str | None)
    """
    results_all   = []
    rank_priority = {"PreferredRank": 2, "NormalRank": 1, "DeprecatedRank": 0}

    sparql = SPARQLWrapper(WIKIDATA_ENDPOINT)
    sparql.setReturnFormat(JSON)
    sparql.setMethod("POST")

    grouped_claims: dict[str, list] = defaultdict(list)
    for c in claims:
        grouped_claims[c["property_id"]].append(c)

    for prop_id, prop_claims in grouped_claims.items():

        for batch_start in range(0, len(prop_claims), batch_size):
            batch = prop_claims[batch_start:batch_start + batch_size]

            # ── De-duplicate (entity, sparql_term) pairs ──────────────────────
            seen_keys     : set                   = set()
            unique_pairs  : list[tuple[str, str]] = []
            key_to_claims : dict                  = defaultdict(list)
            skipped_items : list                  = []

            for c in batch:
                try:
                    sparql_val = format_value(c["value_id"], c.get("value_type"))
                    key        = (c["entity"], sparql_val)

                    if key not in seen_keys:
                        seen_keys.add(key)
                        unique_pairs.append(key)

                    key_to_claims[key].append(c)
                    c["validation_skipped"] = False
                    c["skip_reason"]        = None

                except Exception as exc:
                    c["validation_skipped"] = True
                    c["skip_reason"]        = str(exc)
                    skipped_items.append(c)
                    print(f"⚠️ Skipped: {c['entity']} {prop_id} {c.get('value_id')} | {exc}")

            if not unique_pairs:
                print(f"⚠️ Batch {batch_start // batch_size + 1} — all items invalid, skipping")
                results_all.extend(skipped_items)
                continue

            # ── Build VALUES clause ───────────────────────────────────────────
            values_str = "\n".join(
                f"(wd:{entity} {sparql_val})"
                for entity, sparql_val in unique_pairs
            )

            query = f"""
            PREFIX wd:       <http://www.wikidata.org/entity/>
            PREFIX p:        <http://www.wikidata.org/prop/>
            PREFIX ps:       <http://www.wikidata.org/prop/statement/>
            PREFIX wikibase: <http://wikiba.se/ontology#>

            SELECT ?entity ?value_input ?rank
            WHERE {{
              VALUES (?entity ?value_input) {{
                {values_str}
              }}

              ?entity p:{prop_id} ?statement .
              ?statement ps:{prop_id} ?value .

              FILTER(STR(?value) = STR(?value_input))

              OPTIONAL {{ ?statement wikibase:rank ?rank . }}
            }}
            """

            sparql.setQuery(query)
            results = {"results": {"bindings": []}}

            # ── Retry with exponential backoff ────────────────────────────────
            for attempt in range(5):
                try:
                    results = sparql.query().convert()
                    break
                except Exception as exc:
                    err_msg = str(exc)
                    # Malformed query is permanent — retrying wastes time and
                    # burns rate limit quota, so skip immediately.
                    if "QueryBadFormed" in err_msg or "MalformedQuery" in err_msg:
                        print(f"❌ Malformed query, skipping batch: {exc}")
                        print("❌ VALUES snippet:\n", values_str[:500])
                        break
                    wait = delay * (2 ** attempt)   # 2s, 4s, 8s, 16s, 32s
                    print(f"⚠️ Retry {attempt + 1}/5... {exc}")
                    if attempt == 4:
                        print("❌ VALUES snippet:\n", values_str[:500])
                    time.sleep(wait)

            bindings = results.get("results", {}).get("bindings", [])

            # ── Build result map: (qid, plain_value) → best-ranked hit ────────
            result_map: dict[tuple, dict] = {}

            for row in bindings:
                entity_qid    = row["entity"]["value"].split("/")[-1]
                val_input_key = normalize_value(row["value_input"]["value"])
                rank_uri      = row.get("rank", {}).get("value", "")
                rank_str      = rank_uri.split("#")[-1] if rank_uri else "NormalRank"

                existing = result_map.get((entity_qid, val_input_key))
                if existing is None or \
                   rank_priority.get(rank_str, 0) > rank_priority.get(existing["rank"], 0):
                    result_map[(entity_qid, val_input_key)] = {"exists": True, "rank": rank_str}

            # ── Apply results back to each claim ──────────────────────────────
            for (entity, sparql_val), claims_list in key_to_claims.items():
                lookup_key = _sparql_val_to_lookup_key(sparql_val)

                hit = result_map.get((entity, lookup_key), {"exists": False, "rank": None})

                for c in claims_list:
                    c["triple_exists"] = hit["exists"]
                    c["triple_rank"]   = hit["rank"]
                    c["ref_count"]     = c.get("ref_count", 0)
                    results_all.append(c)

            results_all.extend(skipped_items)

            n_found = sum(1 for v in result_map.values() if v["exists"])
            print(f"✅ {prop_id} batch {batch_start // batch_size + 1} — "
                  f"{len(unique_pairs)} unique, {n_found} found, "
                  f"{len(skipped_items)} skipped")

            time.sleep(delay)

    return results_all

In [ ]:
print("🧪 Validating RDF triples (batch mode)...")
start_time = time.time()
claims = validate_triples_batch(
    claims,
    batch_size=50,
    delay=2.0
)
end_time = time.time()

total        = len(claims)
exists_count = sum(1 for c in claims if c["triple_exists"])
not_found    = [c for c in claims if not c["triple_exists"]]
ranked_count = sum(1 for c in claims if c["triple_rank"] not in [None, "NormalRank"])
pct          = exists_count / total * 100 if total else 0

print("\n📊 Summary:")
print(f"Total claims     : {total}")
print(f"Triple exists    : {exists_count} / {total} ({pct:.2f}%)")
print(f"Triple NOT found : {len(not_found)}")
print(f"Non-normal rank  : {ranked_count}")
print(f"⏱️ Time taken    : {end_time - start_time:.2f}s")

print("\n🔍 Sample results:")
for c in claims[:5]:
    print(f"- {c['claim']}")
    print(f"  exists={c['triple_exists']}, rank={c['triple_rank']}, ref={c['ref_count']}")

print("\n❌ Triples NOT found:")
for c in not_found:
    print(f"- {c['claim']}")
    print(f"  exists={c['triple_exists']}, rank={c['triple_rank']}, ref={c['ref_count']}")
print(f"\nTotal not found: {len(not_found)}")

# Knowledge Graph

# Build Claim-Source Graph

In [26]:
def build_graph(claims):
    G = nx.DiGraph()

    for i, c in enumerate(claims):
        claim_id = f"{c['entity']}_claim_{i}"

        # --- Claim node ---
        G.add_node(
            claim_id,
            type="claim",
            entity=c["entity"],
            claim=c.get("claim"),
            prop_id=c.get("property_id"),
            ref_count=c.get("ref_count"),
            # KG info
            kg_trust=None,
            triple_exists=c.get("triple_exists"),
            triple_rank=c.get("triple_rank"),
            validation_skipped=c.get("validation_skipped", False),
            skip_reason=c.get("skip_reason"),
            # Qualifiers (flattened directly onto the node)
            point_in_time=c.get("qualifiers", {}).get("point_in_time", []),
            start_time=c.get("qualifiers", {}).get("start_time", []),
            end_time=c.get("qualifiers", {}).get("end_time", []),
            # Derived / scoring fields (populated in later pipeline stages)
            latest_date=None,
            date_type = None,
            similarityHybrid=None,
            similarityTFIDF=None,
            similaritySbert=None,
            similarity=None,
            trustworthinessLLM=None,
            timeliness=None,
            trustworthiness=None,
            label=None,
            trustworthinessEqualWeight=None,
            labelEqualWeight=None
        )

        sources = c.get("sources", [])

        if sources:
            for ref in sources:
                ref_id   = ref["ref_id"]
                ref_type = ref.get("ref_type", "unknown")

                if not G.has_node(ref_id):
                    G.add_node(
                        ref_id,
                        type="source",

                        # Reference classification — drives graph rendering and scoring
                        # Possible values: "url" | "stated_in" | "imported_from_wikimedia" | "unknown"
                        ref_type=ref_type,

                        # Reference URL (P854)
                        url=ref.get("url"),

                        # Stated in (P248)
                        stated_in_id=ref.get("stated_in_id"),
                        stated_in_label=ref.get("stated_in_label"),
                        stated_in_website=ref.get("stated_in_website"),

                        # Publisher
                        publisher_label=ref.get("publisher_label"),
                        publisher_website=ref.get("publisher_website"),

                        # Dates
                        retrieved=ref.get("retrieved"),
                        published=ref.get("published"),
                        archive_date=ref.get("archive_date"),
                        archive_url=ref.get("archive_url"),

                        # P143: imported from Wikimedia project
                        # e.g. imported_from="Q328", imported_from_label="English Wikipedia"
                        imported_from=ref.get("imported_from"),
                        imported_from_label=ref.get("imported_from_label"),

                        # Fields populated in later pipeline stages
                        text=None,
                        source_scrape_status=None,
                        source_trust_score=None,
                        pagerank=None,
                        degree=None,
                    )

                G.add_edge(claim_id, ref_id, relation="cited_by")

        else:
            # No references found — create a placeholder node to keep the graph connected
            missing_id = f"no_source_{i}"
            G.add_node(missing_id, type="missing_source")
            G.add_edge(claim_id, missing_id, relation="missing_source")

    return G

In [28]:
# This function is used to traverse the graph per entity to support llm
def get_entity_subgraph_nodes(G, entity_id):

    return [
        node for node, data in G.nodes(data=True)
        if data.get("type") == "claim"
        and data.get("entity") == entity_id
    ]

In [29]:
# Build and Visualize Graph
print("\n🧠 Building graph...")
G_cs = build_graph(claims)
print(f"📊 Graph: {G_cs.number_of_nodes()} nodes, {G_cs.number_of_edges()} edges")
# visualize_graph(G_cs) 


🧠 Building graph...
📊 Graph: 7 nodes, 6 edges


In [ ]:
def visualize_graph_interactive(G):
    net = Network(
        height="750px",
        width="100%",
        directed=True,
        bgcolor="#ffffff",
        font_color="black",
        notebook=True,
        cdn_resources="in_line"
    )

    for node, data in G.nodes(data=True):
        node_type = data.get("type")

        if node_type == "claim":
            label = "CLAIM"
            title = (
                f"Claim: {data.get('claim')}\n"
                f"Timeliness: {data.get('timeliness')}\n"
                f"Reference Count: {data.get('ref_count')}"
            )
            color = "#87CEEB"  # sky blue

        elif node_type == "source":
            ref_type = data.get("ref_type", "unknown")

            # --- Label: use the most descriptive available field ---
            if data.get("stated_in_label"):
                label = data["stated_in_label"]
            elif data.get("imported_from_label"):
                label = data["imported_from_label"]   # e.g. "English Wikipedia"
            elif data.get("url"):
                label = "REF (URL)"
            else:
                label = "REF"

            # --- Tooltip ---
            title = (
                f"Type: {ref_type}\n"
                f"URL: {data.get('url') or '-'}\n"
                f"Stated In: {data.get('stated_in_label') or '-'}"
                f" ({data.get('stated_in_id') or '-'})\n"
                f"Stated In Website: {data.get('stated_in_website') or '-'}\n"
                f"Publisher: {data.get('publisher_label') or '-'}\n"
                f"Publisher Website: {data.get('publisher_website') or '-'}\n"
                f"Imported From: {data.get('imported_from_label') or '-'}"
                f" ({data.get('imported_from') or '-'})\n"   # e.g. "English Wikipedia (Q328)"
                f"Retrieved: {data.get('retrieved') or '-'}\n"
                f"Published: {data.get('published') or '-'}\n"
                f"Archive Date: {data.get('archive_date') or '-'}\n"
                f"Archive URL: {data.get('archive_url') or '-'}\n"
                f"Trust Score: {data.get('source_trust_score')}"
            )

            # --- Color by ref_type ---
            # green  → strong references (direct URL or stated in a known source)
            # orange → weak reference (imported from Wikimedia, not a primary source)
            # grey   → unclassified / empty reference (still attached to keep graph intact)
            color_map = {
                "stated_in":               "#90EE90",  # green
                "url":                     "#90EE90",  # green
                "imported_from_wikimedia": "#FFA500",  # orange — weak, secondary source
                "unknown":                 "#D3D3D3",  # light grey — unclassified
            }
            color = color_map.get(ref_type, "#90EE90")

        else:  # missing_source
            label = "NO SOURCE"
            title = "No reference found"
            color = "#FA8072"  # salmon red

        net.add_node(node, label=label, title=title, color=color)

    for u, v, d in G.edges(data=True):
        net.add_edge(u, v, title=d.get("relation", ""))

    net.show("fact_check_graph.html")

In [ ]:
def kg_trust_score(ref_count, rank):

    score = 0.0

    if ref_count:
        score = 1 - math.exp(-0.3 * ref_count)

    if rank and "PreferredRank" in rank:
        score += 0.1

    elif rank and "DeprecatedRank" in rank:
        score -= 0.3

    return max(0.0, min(score, 1.0))

for node, data in G_cs.nodes(data=True):
    if data.get("type") == "claim":
        G_cs.nodes[node]["kg_trust"] = kg_trust_score(
            ref_count          = data.get("ref_count", 0),
            rank               = data.get("triple_rank"),
        )


count = 0
for node, data in G_cs.nodes(data=True):

    if data.get("type") == "claim":
        print(data.get("claim"))
        print(id(G_cs))
        print("  KG Trust Score:", data.get("kg_trust"))
        print()

        count += 1
        if count >= 5:
            break

# Read Information/Claim as Text

In [16]:
async def fetch_html_playwright_async(url, max_retries=3):

    if not url:
        return None

    BLOCKED_KEYWORDS = [
        "just a moment",
        "enable javascript",
        "checking your browser",
        "access denied",
        "cf-ray",
        "captcha"
    ]

    try:
        async with async_playwright() as p:

            browser = await p.chromium.launch(
                headless=True,
                args=["--no-sandbox", "--disable-dev-shm-usage"]
            )

            context = await browser.new_context(
                user_agent=(
                    "Mozilla/5.0 "
                    "(compatible; MyResearchBot/1.0; "
                    "+https://yourdomain.com/bot)"
                ),
                viewport={"width": 1280, "height": 800},
                java_script_enabled=True,
                locale="en-US"
            )

            async def route_handler(route):
                if route.request.resource_type in ["image", "media", "font"]:
                    await route.abort()
                else:
                    await route.continue_()

            # Block unnecessary resources to improve loading performance
            await context.route("**/*", route_handler)

            page = await context.new_page()

            for attempt in range(max_retries):
                try:
                    print(f"⚙️ Opening page (attempt {attempt + 1})...")

                    await page.goto(
                        url,
                        timeout=60000,
                        wait_until="domcontentloaded"
                    )

                    try:
                        await page.wait_for_load_state(
                            "networkidle",
                            timeout=10000
                        )
                    except:
                        # Fallback delay if the page never reaches network idle
                        await page.wait_for_timeout(2000)

                    # IMDb relies heavily on expandable UI elements.
                    # Expand hidden sections such as "See more" to
                    # retrieve more complete page content.
                    is_imdb = "imdb.com" in url.lower()

                    if is_imdb:
                        print("🎬 IMDb detected → Expanding hidden content...")

                        await expand_hidden_content(page)

                        try:
                            await page.wait_for_load_state(
                                "networkidle",
                                timeout=5000
                            )
                        except:
                            await page.wait_for_timeout(1500)

                    # Retrieve the final rendered HTML
                    html = await page.content()
                    html_lower = html.lower()

                    # Detect common anti-bot or access restriction pages
                    if any(k in html_lower for k in BLOCKED_KEYWORDS):
                        print("🚫 Blocked by site protection")
                        await browser.close()
                        return None

                    print("✅ Page loaded:", len(html))

                    await browser.close()
                    return html

                except Exception as e:
                    print(f"⚠️ Attempt {attempt + 1} failed:", e)
                    await asyncio.sleep(2 ** attempt)

            await browser.close()
            return None

    except Exception as e:
        print("❌ Playwright fatal error:", e)
        return None

In [17]:
def fetch_html_playwright(url):
    return asyncio.get_event_loop().run_until_complete(
        fetch_html_playwright_async(url)
    )

In [18]:
# soup = objek hasil parsing HTML dengan BeautifulSoup
def extract_jsonld_dates(soup):

    scripts = soup.find_all("script", type="application/ld+json")

    for script in scripts:
        try:
            data = json.loads(script.string)

            if isinstance(data, list):
                for item in data:
                    if isinstance(item, dict):
                        if item.get("datePublished") or item.get("dateModified"):
                            return item.get("datePublished"), item.get("dateModified")

            elif isinstance(data, dict):
                if data.get("datePublished") or data.get("dateModified"):
                    return data.get("datePublished"), data.get("dateModified")

        except:
            continue

    return None, None


def extract_meta_dates(soup):
    meta_keys = [
        "article:published_time",
        "article:modified_time",
        "datePublished",
        "dateModified",
        "pubdate",
        "lastmod",
        "last-modified",  # for Britannica
        "lastModified"   
    ]

    for key in meta_keys:
        tag = (
            soup.find("meta", property=key) or
            soup.find("meta", attrs={"name": key}) or
            soup.find("meta", itemprop=key)
        )
        if tag and tag.get("content"):
            return tag["content"]

    return None

def extract_time_tag(soup):

    times = soup.find_all("time")

    print("DEBUG extract_time_tag found:", len(times))

    for t in times:

        print("DEBUG time tag:", t)

        if t.get("datetime"):
            print("DEBUG datetime:", t["datetime"])
            return t["datetime"]

        txt = t.get_text(strip=True)
        if txt:
            print("DEBUG text:", txt)
            return txt

    return None


def extract_strong_date_patterns(text):

    patterns = [

        # 2026-03-11
        r"\b\d{4}-\d{2}-\d{2}\b",

        # March 11, 2026
        r"\b[A-Z][a-z]+\s\d{1,2},\s\d{4}\b",

        # Mar. 11, 2026
        r"\b[A-Z][a-z]{2}\.\s\d{1,2},\s\d{4}\b",

        # 11 March 2026
        r"\b\d{1,2}\s[A-Z][a-z]+\s\d{4}\b"
    ]

    for p in patterns:
        match = re.search(p, text)
        if match:
            return match.group()

    return None


def extract_copyright_year(text):
    pattern = r"""
        (?:©|[Cc]opyright\s*©?)
        \s*
        (\d{4})
        (?:\s*[-–,]\s*\d{4})*
    """
    match = re.search(pattern, text, re.VERBOSE)
    if not match:
        return None

    all_years = re.findall(r'\d{4}', match.group(0))
    return max(all_years)

In [19]:
def extract_rating_from_badge(html):
    """
    Extract content ratings from visual badges such as BBFC, FSK, or MPAA.

    Supported cases:
    - <img alt="12 certificate"> or <img alt="BBFC 12">
    - <span aria-label="Rated 12">
    - <div class="certificate">12</div>
    - SVG elements containing rating text
    - data-testid="certificate-rating"
    """
    soup = BeautifulSoup(html, "html.parser")
    results = []

    # ============================================================
    # PATTERN 1: Rating stored in image alt/title attributes
    # <img alt="12 certificate">
    # <img alt="Rated PG-13">
    # ============================================================
    rating_alt_pattern = re.compile(
        r'(?:rated?|certificate|bbfc|mpaa|pegi|fsk|avf)?'
        r'\s*'
        r'(u|pg|12a|12|15|18|r|g|pg-13|nc-17|tv-ma|\d{1,2})'
        r'\s*'
        r'(?:\+|certificate|rated?|bbfc)?',
        re.IGNORECASE
    )

    for img in soup.find_all("img"):
        alt = img.get("alt", "") + " " + img.get("title", "")
        match = rating_alt_pattern.search(alt)

        if match:
            results.append(f"content rating {match.group(1)}")

    # ============================================================
    # PATTERN 2: Rating stored in aria-label attributes
    # <span aria-label="Rated 12">
    # ============================================================
    for tag in soup.find_all(attrs={"aria-label": True}):
        label = tag["aria-label"]
        match = rating_alt_pattern.search(label)

        if match:
            results.append(f"content rating {match.group(1)}")

    # ============================================================
    # PATTERN 3: Elements related to certificate or rating labels
    # <div class="certificate">12</div>
    # <span data-testid="certificate-rating">12</span>
    # ============================================================
    cert_selectors = [
        {"data-testid": re.compile(r"certif|rating|bbfc", re.I)},
        {"class": re.compile(r"certif|rating-badge|age-rating|bbfc", re.I)},
        {"itemprop": "contentRating"},
    ]

    for attrs in cert_selectors:
        for tag in soup.find_all(attrs=attrs):
            text = tag.get_text(strip=True)
            match = rating_alt_pattern.search(text)

            if match:
                results.append(f"content rating {match.group(1)}")

    # ============================================================
    # PATTERN 4: Rating stored inside SVG text nodes
    # <svg><text>12</text></svg>
    # ============================================================
    for svg_text in soup.find_all("text"):
        content = svg_text.get_text(strip=True)

        match = re.match(
            r'^(u|pg|12a|12|15|18|\d{1,2})$',
            content,
            re.I
        )

        if match:
            results.append(f"content rating {match.group(1)}")

    # ============================================================
    # PATTERN 5: JSON-LD structured metadata
    # {"contentRating": "12"}
    # ============================================================
    for script in soup.find_all(
        "script",
        type="application/ld+json"
    ):
        try:
            import json

            data = json.loads(script.string)

            rating = (
                data.get("contentRating")
                or data.get("ratingValue")
            )

            if rating:
                results.append(f"content rating {rating}")

        except Exception:
            pass

    return list(set(results))


def enrich_text(full_text, html):
    """
    Append extracted metadata into the scraped text content.
    """
    enriched = extract_rating_from_badge(html)

    if enriched:
        full_text += "\n\n[METADATA]\n" + "\n".join(enriched)

    return full_text

- Aria Label adalah atribut html untuk memberikan label teks deskriptif pada halaman web, terutama yang hanya menampilkan ikon tanpa teks (misal button)

- SVG = Scalable Vector Graphics

Ini adalah format gambar berbasis vector (XML), bukan pixel.

In [20]:
def detect_invalid_goid_page(soup):
    main = soup.find("main")

    if not main:
        return True

    # Extract main visible text content
    main_text = main.get_text(" ", strip=True)

    # Count common content-related elements
    content_tags = main.find_all(
        ["p", "div", "section", "article"]
    )

    # Count direct child elements inside <main>
    direct_children = main.find_all(recursive=False)

    # Indicators of an empty or improperly rendered SPA page
    conditions = [
        len(main_text) < 80,       # Text content is too short
        len(content_tags) < 3,     # Very few content elements found
        len(direct_children) == 0  # Empty main container
    ]

    # Consider the page invalid if at least two conditions are met
    if sum(conditions) >= 2:
        return True

    return False

In [21]:
# Validation after playwright fetch
def is_unreachable(text):

    if not text:
        return True

    if len(text.strip()) < 200:
        return True

    suspicious_keywords = [
        "enable javascript", # playwright blocked
        "just a moment",
        "checking your browser",
        "access denied",
        "cf-ray",
        "captcha"
    ]

    text_lower = text.lower()

    if any(k in text_lower for k in suspicious_keywords):
        return True

    return False

In [22]:
def need_playwright(res, text):
    html = res.text.lower()
    text_lower = text.lower()

    signals = [
        len(text) < 500,

        # UI interaction
        "see more" in html,
        "load more" in html,
        "expand" in html,

        # framework
        "__next" in html,

        # testing attribute (weak signal), there could be information in it example rating badge di imdb
        "data-testid" in html,

        # 🚨 BLOCK / JS REQUIRED (STRONG SIGNAL)
        "enable javascript" in text_lower,
        "just a moment" in text_lower,
        "checking your browser" in text_lower,
        "access denied" in text_lower,
        "captcha" in text_lower
    ]

    return sum(signals) >= 2

In [23]:
def is_blocked_page(text):

    if not text:
        return False

    blocked_signals = [
        "attention required",
        "cloudflare",
        "access denied",
        "error 522",
        "error 1020",
        "you have been blocked",
        "enable cookies"
    ]

    text_lower = text.lower()

    return any(k in text_lower for k in blocked_signals)

In [24]:
def crawl_text(url):

    if not url:
        return {"page_status": "invalid_url"}

    try:
        print("\n==============================")
        print("🌐 Crawling:", url)
        print("==============================")

        # ── FILE TYPE DETECTION (NEW) ────────────────────────────
        file_ext = url.split("?")[0].split(".")[-1].lower()
        non_html_ext = ["pdf", "csv", "xlsx", "xls", "doc", "docx", "ppt", "pptx", "zip", "rar"]

        if file_ext in non_html_ext:
            return {
                "page_status": "file_type",
                "file_type": file_ext,
                "text": None
            }

        # ── HEADER ───────────────────────────────────────────────
        headers = {
            "User-Agent": "Mozilla/5.0",
            "Accept-Language": "en-US,en;q=0.9",
        }

        session = requests.Session()
        session.headers.update(headers)

        res = session.get(url, timeout=20)

        print("HTTP STATUS:", res.status_code)
        print("HTML length:", len(res.text))

        # ── HTTP STATUS ──────────────────────────────────────────
        KNOWN_HTTP_STATUSES = {
            400: "bad_request",
            401: "unauthorized",
            403: "forbidden",
            404: "not_found",
            405: "method_not_allowed",
            410: "gone",
            429: "too_many_requests",
        }

        status = res.status_code
        
        if status != 200:
            if status in KNOWN_HTTP_STATUSES:
                page_status = KNOWN_HTTP_STATUSES[status]
            elif status >= 500:
                page_status = "server_error"
            else:
                page_status = f"http_{status}"
            
            return {"page_status": page_status, "http_status": status}

        # ── DEFAULT VALUES ───────────────────────────────────────
        html        = res.text
        full_text   = ""
        page_status = "success"
        soup        = None

        # ── DECISION: Playwright directly? ───────────────────────
        content_type = res.headers.get("Content-Type", "")
        is_empty     = len(res.text.strip()) == 0
        is_not_html  = "text/html" not in content_type

        if is_not_html:
            return {
                "page_status": "file_type",
                "content_type": content_type,
                "text": None
            }

        if is_empty:
            print("Empty body → forcing Playwright")
            use_playwright = True

        else:
            soup = BeautifulSoup(html, "html.parser")

            if "indonesia.go.id" in url:
                if detect_invalid_goid_page(soup):
                    print("invalid go.id route detected")
                    page_status = "invalid_goid_page"

            for s in soup(["script", "style"]):
                s.decompose()

            full_text      = soup.get_text(" ", strip=True)
            use_playwright = need_playwright(res, full_text)

        # ── PLAYWRIGHT ───────────────────────────────────────────
        if use_playwright:
            print("⚙️ Switching to Playwright (JS rendering / hidden content detected)...")

            try:
                html_pw = asyncio.run(fetch_html_playwright_async(url))
            except RuntimeError:
                html_pw = asyncio.get_event_loop().run_until_complete(
                    fetch_html_playwright_async(url)
                )

            if html_pw:
                soup = BeautifulSoup(html_pw, "html.parser")

                for s in soup(["script", "style"]):
                    s.decompose()

                full_text = soup.get_text(" ", strip=True)
                html      = html_pw

                print("✅ Playwright content loaded:", len(full_text))

            else:
                return {"page_status": "blocked", "text": None}

        # ── VALIDASI KONTEN ──────────────────────────────────────
        if is_unreachable(full_text):
            return {"page_status": "unknown", "text": None}

        # ── ENRICHMENT ───────────────────────────────────────────
        full_text = enrich_text(full_text, html)

        # ── DATE EXTRACTION ──────────────────────────────────────
        json_pub, json_mod = extract_jsonld_dates(soup)
        meta_date          = extract_meta_dates(soup)
        time_date          = extract_time_tag(soup)
        pattern_date       = extract_strong_date_patterns(full_text)
        copyright_year     = extract_copyright_year(full_text)
        last_modified      = res.headers.get("Last-Modified")

        print(f"✅ Crawl success | chars: {len(full_text)} | preview: {full_text[:200]}...")
        print(f"📅 Dates → published: {json_pub} | modified: {json_mod} | meta: {meta_date} | time_tag: {time_date} | pattern: {pattern_date}")

        return {
            "page_status":          page_status,
            "text":                 full_text,
            "preview":              full_text[:500],
            "published_date":       json_pub,
            "modified_date":        json_mod,
            "meta_date":            meta_date,
            "time_tag":             time_date,
            "pattern_date":         pattern_date,
            "last_modified_header": last_modified,
            "copyright_year":       copyright_year,
        }

    except requests.exceptions.Timeout:
        return {"page_status": "timeout"}

    except requests.exceptions.ConnectionError:
        return {"page_status": "connection_error"}

    except Exception as e:
        print("crawl error:", e)
        return {"page_status": "crawl_error"}

In [30]:
# Iterasi all node, call function crawl_text
for node, data in G_cs.nodes(data=True):
    # Only process source nodes
    if data.get("type") != "source":
        continue

    # Skip source nodes that have no URL (e.g. stated_in only, no reference URL)
    url = data.get("url")
    if not url:
        continue

    # Skip already successfully crawled sources to avoid redundant requests
    if data.get("source_scrape_status") == "success":
        continue

    print("🌐 Crawling for TEXT:", url)
    crawl_data = crawl_text(url)
    if not crawl_data:
        continue

    # Store crawled text and status directly into the source node attributes
    if crawl_data.get("page_status") == "success":
        data["text"]          = crawl_data.get("text")
        data["source_scrape_status"] = "success"
        print("✅ TEXT STORED in SOURCE:", url)

🌐 Crawling for TEXT: https://www.un.org/en/about-us/un-charter

🌐 Crawling: https://www.un.org/en/about-us/un-charter
HTTP STATUS: 200
HTML length: 96728
DEBUG extract_time_tag found: 0
✅ Crawl success | chars: 7641 | preview: UN Charter | United Nations Skip to main content Toggle navigation Welcome to the United Nations العربية 中文 Nederlands English Français Deutsch Kreyòl हिन्दी Bahasa Indonesia Italiano Polski Português...
📅 Dates → published: None | modified: None | meta: None | time_tag: None | pattern: 26 June 1945
✅ TEXT STORED in SOURCE: https://www.un.org/en/about-us/un-charter
🌐 Crawling for TEXT: https://www.un.org/es/about-us/un-charter

🌐 Crawling: https://www.un.org/es/about-us/un-charter
HTTP STATUS: 200
HTML length: 97161
DEBUG extract_time_tag found: 0
✅ Crawl success | chars: 8984 | preview: Carta de las Naciones Unidas | Naciones Unidas Skip to main content Toggle navigation Bienvenidos a las Naciones Unidas العربية 中文 Nederlands English Français Deutsch Kreyòl हिन्

# Ensure Source Has Published Date

In [31]:
# Make sure crawl_text already populated time-related properties
for node, data in G_cs.nodes(data=True):
    if data.get("type") != "claim":
        continue

    # Reset latest_date and date_type for this claim
    latest_date = None
    latest_date_type = None

    G_cs.nodes[node]["latest_date"] = None
    G_cs.nodes[node]["date_type"] = None

    # Get all source nodes connected to this claim
    sources = [
        s for s in G_cs.successors(node)
        if G_cs.nodes[s].get("type") == "source"
    ]
    if not sources:
        continue

    print("\n==============================")
    print("CLAIM:", data.get("claim"))
    print("==============================")

    for src_node in sources:
        src_data = G_cs.nodes[src_node]
        url = src_data.get("url")

        # Skip source nodes with no URL
        if not url:
            continue

        print("🌐 Crawling:", url)

        # Use cached crawl result if available to avoid redundant requests
        if src_data.get("crawl_data"):
            crawl_data = src_data["crawl_data"]
            print("FROM CACHE:", url)
        else:
            crawl_data = crawl_text(url)
            src_data["crawl_data"] = crawl_data

        print("DEBUG CRAWL DATA:", crawl_data)

        date_value = None
        date_type = None

        # Priority 1: use Wikidata qualifier dates from the source node
        if src_data.get("published"):
            date_value = src_data["published"]
            date_type = "published (wikidata)"

        elif src_data.get("retrieved"):
            date_value = src_data["retrieved"]
            date_type = "retrieved (wikidata)"

        # Priority 2: fallback to dates extracted from crawling
        else:
            if crawl_data:
                if crawl_data.get("published_date"):
                    date_value = crawl_data["published_date"]
                    date_type = "crawl:jsonld_published"

                elif crawl_data.get("modified_date"):
                    date_value = crawl_data["modified_date"]
                    date_type = "crawl:jsonld_modified"

                elif crawl_data.get("meta_date"):
                    date_value = crawl_data["meta_date"]
                    date_type = "crawl:meta"

                elif crawl_data.get("time_tag"):
                    date_value = crawl_data["time_tag"]
                    date_type = "crawl:time_tag"

                elif crawl_data.get("pattern_date"):
                    date_value = crawl_data["pattern_date"]
                    date_type = "crawl:pattern"

                elif crawl_data.get("last_modified_header"):
                    date_value = crawl_data["last_modified_header"]
                    date_type = "crawl:http_last_modified"

                elif crawl_data.get("copyright_year"):
                    date_value = crawl_data["copyright_year"]
                    date_type = "crawl:copyright"

                else:
                    date_value = "Unknown"
                    date_type = "unknown"

        # Parse date and track the latest one across all sources for this claim
        try:
            if date_value and date_value != "Unknown":
                parsed = parser.parse(date_value)

                if latest_date is None or parsed > latest_date:
                    latest_date = parsed
                    latest_date_type = date_type

        except Exception:
            pass

        print("link url:", url)
        print(f"date: {date_value} ({date_type})")
        print()

    # Assign the latest date found to the claim node
    if latest_date:
        G_cs.nodes[node]["latest_date"] = latest_date
        G_cs.nodes[node]["date_type"] = latest_date_type


CLAIM: United Nations — main regulatory text: Charter of the United Nations
🌐 Crawling: https://www.un.org/en/about-us/un-charter

🌐 Crawling: https://www.un.org/en/about-us/un-charter
HTTP STATUS: 200
HTML length: 96728
DEBUG extract_time_tag found: 0
✅ Crawl success | chars: 7641 | preview: UN Charter | United Nations Skip to main content Toggle navigation Welcome to the United Nations العربية 中文 Nederlands English Français Deutsch Kreyòl हिन्दी Bahasa Indonesia Italiano Polski Português...
📅 Dates → published: None | modified: None | meta: None | time_tag: None | pattern: 26 June 1945
DEBUG CRAWL DATA: {'page_status': 'success', 'text': "UN Charter | United Nations Skip to main content Toggle navigation Welcome to the United Nations العربية 中文 Nederlands English Français Deutsch Kreyòl हिन्दी Bahasa Indonesia Italiano Polski Português Русский Español Kiswahili Türkçe Українська Peace, dignity and equality on a healthy planet Search the United Nations Submit Search A-Z Site Index To

# Feature Engineering (trustworthiness, timeliness, similarity)

## Helper Function Switch Model

In [ ]:
# Get API key for kaggle
user_secrets = UserSecretsClient()
client = Groq(api_key=user_secrets.get_secret("GROQ_API_KEY"))

# Get API key for colab
# from google.colab import userdata

# client = Groq(
#     api_key=userdata.get("GROQ_API_KEY")
# )

# Get API key for local
# from dotenv import load_dotenv
# import os

# load_dotenv()

# client = Groq(
#     api_key=os.getenv("GROQ_API_KEY")
# )

# Ordered models: cheaper → more expensive
MODELS = [
    "llama-3.1-8b-instant",
    "llama-3.3-70b-versatile",
    "openai/gpt-oss-120b"
]


def call_with_fallback(prompt):
    """
    Try multiple models sequentially until one succeeds.

    Returns:
        tuple: (response_text, model_used)
    """
    last_error = None

    for model in MODELS:
        try:
            print(f"⚙️ Trying model: {model}")

            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0
            )

            print(f"✅ Success with: {model}")

            # Return BOTH response and model used
            return response.choices[0].message.content.strip(), model

        except Exception as e:
            error_str = str(e)

            # Rate limit → switch to next model
            if "rate_limit" in error_str or "429" in error_str:
                print(f"⛔ Rate limit on {model}, switching...")
                last_error = e
                continue

            # Timeout → retry after short delay
            elif "timeout" in error_str.lower():
                print(f"⏳ Timeout on {model}, retrying...")
                time.sleep(2)
                continue

            # Other errors → stop execution
            else:
                raise e

    raise Exception(f"❌ All models failed. Last error: {last_error}")

# Classify Category Decay of information and Check if the Source Support the Claim Using LLM

In [ ]:
# Wikidata API base endpoint — used for all entity/property lookups
WIKIDATA_API = "https://www.wikidata.org/w/api.php"

def fetch_properties_batch(prop_ids, lang="en"):
    """
    Fetch labels and descriptions for multiple Wikidata properties in bulk.

    Sends up to 50 property IDs per API request (Wikidata's wikibase get entities limit),
    chunking larger lists automatically. Each chunk is retried up to 3 times with
    exponential backoff on failure. A 1-second delay is added between chunks to
    avoid rate limiting.

    Preferred over per-property fetching because it reduces total API calls and
    network overhead significantly when processing a knowledge graph with many
    distinct properties.

    Args:
        prop_ids (list[str]): List of Wikidata property IDs, e.g. ["P31", "P150", "P47"]
        lang (str): Language code for label/description (default: "en")

    Returns:
        dict: Mapping of property ID to its metadata, e.g.
              {
                  "P31": {"label": "instance of", "description": "..."},
                  "P150": {"label": "contains ...", "description": "..."}
              }
              Properties that fail all retries are omitted from the result.
    """
    results = {}

    # Wikidata wbgetentities supports a maximum of 50 IDs per request
    chunk_size = 50

    for i in range(0, len(prop_ids), chunk_size):
        chunk = prop_ids[i:i + chunk_size]

        params = {
            "action": "wbgetentities",
            "ids": "|".join(chunk),       # pipe-separated list of property IDs
            "format": "json",
            "languages": lang,
            "props": "labels|descriptions" # only fetch what we need to minimize response size
        }

        max_retries = 3
        for attempt in range(max_retries):
            try:
                response = requests.get(WIKIDATA_API, params=params, timeout=15)

                # Empty body or non-200 usually indicates rate limiting or a transient server error
                if response.status_code != 200 or not response.text.strip():
                    print(f"⚠️ HTTP {response.status_code}, attempt {attempt+1}")
                    time.sleep(2 ** attempt)  # exponential backoff: 1s, 2s, 4s
                    continue

                data = response.json()

                # Extract label and description for each property in this chunk
                for prop_id in chunk:
                    entity = data.get("entities", {}).get(prop_id, {})
                    label = entity.get("labels", {}).get(lang, {}).get("value", "")
                    description = entity.get("descriptions", {}).get(lang, {}).get("value", "")
                    results[prop_id] = {"label": label, "description": description}
                    print(f"✅ {prop_id} | {label} | {description}")

                break  # chunk succeeded, exit retry loop

            except Exception as e:
                print(f"⚠️ Batch fetch error (attempt {attempt+1}): {e}")
                time.sleep(2 ** attempt)  # exponential backoff before retry

        # Pause between chunks to reduce risk of hitting Wikidata rate limits
        if i + chunk_size < len(prop_ids):
            time.sleep(1)

    return results

In [ ]:
UNIFIED_SYSTEM_PROMPT = """You are a precise fact-checking and information decay analyst. You evaluate factual claims against source texts, and classify how quickly real-world information becomes outdated.

For each item, perform TWO tasks:

## TASK 1 — LLM TRUSTWORTHINESS SCORE
Determine how well the source text supports the claim.

Scoring:
1.0 = explicitly supported
0.7–0.9 = strongly implied
0.4–0.6 = partial support
0.1–0.3 = weak support
0.0 = not supported or contradicted

Rules:
- Interpret the relation meaning using the provided property description
- Verify the claim ONLY against the provided source text
- Do NOT use external knowledge to judge whether the claim is true or false
- Cross-language semantic match is VALID (translate mentally)
- Evaluate each pair INDEPENDENTLY — do not let one pair influence another

## TASK 2 — TEMPORAL CATEGORY
Classify how quickly this specific claim becomes outdated.

You MAY and SHOULD use external knowledge to determine realistic decay behavior.

Categories:
- static    → almost never changes (permanent facts)
- slow      → changes over decades
- medium    → changes over months to a few years
- fast      → changes within months
- very_fast → changes within days or hours

Guidelines:
- Use the property description to infer how likely the value is to change
- Distinguish between permanent facts and time-sensitive states
- Derived values can change even if the underlying fact does not
  (e.g., age = fast even though date of birth = static)

Reference patterns:
- static:    birth/death dates, family relations, creation facts, historical events
- slow:      nationality, capital city, official language
- medium:    occupation, population, organization attributes
- fast:      positions held, memberships, relationships
- very_fast: prices, live status, rankings, rapidly changing events

Default to "medium" if uncertain.

## STRICT PROHIBITIONS
- Do NOT output any text outside the JSON array
- Do NOT skip any ID in the response
- Do NOT merge or compare claims across items

## OUTPUT FORMAT
Return ONLY a valid JSON array with ALL IDs present:
[
  {"id": 0, "score": 0.9, "temporal": "static", "reason": "Source text explicitly states X, directly matching the claim."},
  {"id": 1, "score": 0.3, "temporal": "fast", "reason": "Source text mentions related context but does not confirm the specific value."}
]

The "reason" field must be one concise sentence explaining the score."""

## Function to Use the LLM

In [ ]:
def process_entity_llm(G, entity_id, batch_size=8):
    """
    Unified LLM processor:
    - Computes trustworthiness score (LLM-based)
    - Classifies temporal category

    This version:
    - Uses Wikidata property description
    - Does NOT use sentence extraction
    - Uses raw text cutoff (first 600 chars)

    Args:
        G (networkx.DiGraph): Knowledge graph
        entity_id (str): Target entity
        batch_size (int): Number of claim-source pairs per LLM call
    """

    claim_nodes = get_entity_subgraph_nodes(G, entity_id)
    if not claim_nodes:
        return
    
    # Get all unique prop_id
    all_prop_ids = list(set(
        G.nodes[n].get("prop_id")
        for n in claim_nodes
        if G.nodes[n].get("prop_id")
    ))

    # Fetch property's description before processing
    prop_cache = fetch_properties_batch(all_prop_ids)

    items = []

    # STEP 1: Collect claim-source pairs
    for claim_node in claim_nodes:

        claim_text = G.nodes[claim_node].get("claim", "")
        prop_id = G.nodes[claim_node].get("prop_id")

        # Retrieve property metadata
        prop_info = prop_cache.get(prop_id, {})
        prop_label = prop_info.get("label", "")
        prop_desc = prop_info.get("description", "")

        # Get all connected source nodes that successfully crawled text
        sources = [
            s for s in G.successors(claim_node)
            if G.nodes[s].get("type") == "source"
            and G.nodes[s].get("source_scrape_status") == "success"
        ]

        for src_node in sources:
            raw_text = G.nodes[src_node].get("text", "")
            if not raw_text:
                continue

            # Simple truncation (no preprocessing)
            text_cut = raw_text[:600]

            items.append({
                "claim_node": claim_node,
                "source_node": src_node,
                "claim": claim_text,
                "text": text_cut,
                "prop_id": prop_id,
                "prop_label": prop_label,
                "prop_desc": prop_desc,
            })

    if not items:
        return

    # STEP 2: Process in batches
    for i in range(0, len(items), batch_size):
        batch = items[i:i + batch_size]

        # Format batch input for LLM
        pair_list_text = "\n\n".join([
            f"""ID: {idx}
Claim: {p['claim']}
Property ID: {p['prop_id']}
Property label: {p['prop_label']}
Property description: {p['prop_desc']}
Source text: {p['text']}"""
            for idx, p in enumerate(batch)
        ])

        prompt = f"""{UNIFIED_SYSTEM_PROMPT}

---

### INPUT

{pair_list_text}

---

Return ONLY JSON array (IDs 0 to {len(batch) - 1}):"""

        # 🔥 Get both output and model used
        output, model_used = call_with_fallback(prompt)

        print(f" Model used for this batch: {model_used}")

        # STEP 3: Parse LLM response
        try:
            match = re.search(r'\[[\s\S]*\]', output)
            if not match:
                print("⚠️ No JSON detected")
                print(output)
                continue

            data = json.loads(match.group())

            for item in data:
                idx = item.get("id")

                if idx is None or not (0 <= idx < len(batch)):
                    continue

                p = batch[idx]

                # Print per claim along with model used
                print(f"  [{p['claim_node']}] {p['claim'][:60]}... → ✅ success with {model_used}")

                # Assign LLM trust score to edge
                score = max(0.0, min(1.0, float(item.get("score", 0.0))))
                G.edges[p["claim_node"], p["source_node"]]["llm_score"] = score

                # Store explanation if available
                reason = item.get("reason", "")
                if reason:
                    G.nodes[p["source_node"]]["source_trust_reason"] = reason

                # Assign temporal category to claim node
                temporal = item.get("temporal", "medium")
                if temporal in ["very_fast", "fast", "medium", "slow", "static"]:
                    G.nodes[p["claim_node"]]["temporal_category"] = temporal

        except Exception as e:
            print("⚠️ JSON parse error:", e)
            print(output)

In [ ]:
# ─────────────────────────────────────────────
# Determine LLM-based trust scores and temporal categories for ALL claims in the graph
# ─────────────────────────────────────────────

for entity_id in entities:
    process_entity_llm(G_cs, entity_id)

In [ ]:
def compute_llm_trust_from_graph(G, claim_node):
    """
    Aggregate LLM trust scores per claim from edge attributes.
    LLM scores are expected to already be stored on edges by a prior LLM call.
    """
    claim_text = G.nodes[claim_node].get("claim")
    print(f"\n🧪 CLAIM NODE: {claim_node}")
    print(f"📝 Claim: {claim_text}")

    # Get all source nodes connected to this claim
    sources = [
        s for s in G.successors(claim_node)
        if G.nodes[s].get("type") == "source"
    ]
    print(f"🔗 Total sources: {len(sources)}")

    if not sources:
        print("⚠️ No source, score = 0.0")
        return 0.0

    scores = []
    for i, s in enumerate(sources, 1):
        # Read LLM score from edge attribute
        score   = G.edges[claim_node, s].get("llm_score")
        src = (
            G.nodes[s].get("url")
            or G.nodes[s].get("stated_in_label")
            or G.nodes[s].get("publisher_label")
            or s
        )   

        print(f"\n   📄 Source {i}: {src}")

        if score is None:
            print("   ⚠️ No llm score, calculate as 0.0")
            scores.append(0.0)
            continue

        print(f"   🤖 LLM Score: {score}")
        scores.append(score)

    if not scores:
        print("⚠️ No valid score at all, return 0.0")
        return 0.0

    # Average score across all sources
    final_score = sum(scores) / len(scores)
    print(f"\n✅ FINAL SCORE (avg): {final_score}")
    return final_score

In [ ]:
# Calculate LLM trust score for every claim node
for node, data in G_cs.nodes(data=True):

    if data.get("type") != "claim":
        continue

    score = compute_llm_trust_from_graph(G_cs, node)

    G_cs.nodes[node]["trustworthinessLLM"] = score

    print(f"\n🎯 Final Results -> {data.get('claim')} : {score}")  # debug
    print("=" * 80)

# Degree dan cosine similarity

In [ ]:
print("\n📊 Computing centrality metrics...")

# Degree centrality measures how many connections a source has
# relative to all possible connections in the graph.
# NetworkX returns values already normalized to [0, 1].

degree_c = nx.degree_centrality(G_cs)

# Store source-level graph metrics

for node, data in G_cs.nodes(data=True):

    if data.get("type") != "source":
        continue

    G_cs.nodes[node]["degree"] = degree_c.get(node, 0.0)

In [ ]:
# Hitung similarity
def compute_claim_source_similarity(claim, reference):
    vectorizer = TfidfVectorizer()
    tfidf = vectorizer.fit_transform([claim, reference])

    claim_vec = tfidf[0:1]  # claim representation
    ref_vec   = tfidf[1:2]  # reference representation

    similarity_matrix = cosine_similarity(claim_vec, ref_vec)

    # similarity_matrix shape = (1, 1), get the scalar value
    return similarity_matrix[0, 0]

In [ ]:
sbert_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

def compute_sbert_similarity(text_a, text_b):

    emb = sbert_model.encode(
        [text_a, text_b],
        normalize_embeddings=True
    )

    return float(np.dot(emb[0], emb[1]))

In [ ]:
# ------------------------------ Hybrid Similarity ------------------------------
def compute_hybrid_similarity(claim, reference):

    tfidf_sim = compute_claim_source_similarity(
        claim,
        reference
    )

    sbert_sim = compute_sbert_similarity(
        claim,
        reference
    )

    similarity_hybrid = np.mean([
        tfidf_sim,
        sbert_sim
    ])

    return {
        "tfidf": tfidf_sim,
        "sbert": sbert_sim,
        "hybrid": similarity_hybrid
    }

In [ ]:
print("\n🧪 Verifying claims (TF-IDF + SBERT + Hybrid)...\n")

for idx, (node, data) in enumerate(G_cs.nodes(data=True), 1):

    if data.get("type") != "claim":
        continue

    claim_text = data.get("claim")

    print(f"{idx:02d}. Claim: {claim_text}")

    sources = [
        s for s in G_cs.successors(node)
        if G_cs.nodes[s].get("type") == "source"
    ]

    print(f"    Sources found: {len(sources)}")

    tfidf_scores = []
    sbert_scores = []
    hybrid_scores = []

    for s in sources:

        src_text = G_cs.nodes[s].get("text")

        if not src_text:
            continue

        scores = compute_hybrid_similarity(
            claim_text,
            src_text
        )

        tfidf_sim = scores["tfidf"]
        sbert_sim = scores["sbert"]
        hybrid_sim = scores["hybrid"]

        tfidf_scores.append(tfidf_sim)
        sbert_scores.append(sbert_sim)
        hybrid_scores.append(hybrid_sim)

        print(
            f"      - tfidf={tfidf_sim:.3f} "
            f"| sbert={sbert_sim:.3f} "
            f"| hybrid={hybrid_sim:.3f}"
        )

    if tfidf_scores:

        similarityTFIDF = max(tfidf_scores)
        similaritySbert = max(sbert_scores)
        similarityHybrid = max(hybrid_scores)

        similarity = max(
            similarityTFIDF,
            similaritySbert,
            similarityHybrid
        )

        G_cs.nodes[node]["similarityTFIDF"] = similarityTFIDF
        G_cs.nodes[node]["similaritySbert"] = similaritySbert
        G_cs.nodes[node]["similarityHybrid"] = similarityHybrid
        G_cs.nodes[node]["similarity"] = similarity

        print(
            f"    ✅ TF-IDF Best : {similarityTFIDF:.3f}"
        )
        print(
            f"    ✅ SBERT Best  : {similaritySbert:.3f}"
        )
        print(
            f"    ✅ Hybrid Best : {similarityHybrid:.3f}"
        )
        print(
            f"    🏆 Final Best  : {similarity:.3f}"
        )

    else:

        G_cs.nodes[node]["similarityTFIDF"] = 0.0
        G_cs.nodes[node]["similaritySbert"] = 0.0
        G_cs.nodes[node]["similarityHybrid"] = 0.0
        G_cs.nodes[node]["similarity"] = 0.0

        print(
            "    ❌ No valid source text → similarity = 0.0"
        )

    print()

# Feature engineering source trust score (based on pagerank)

In [ ]:
# Get API key for kaggle
user_secrets = UserSecretsClient()
API_KEY = user_secrets.get_secret("OPENPAGERANK_API_KEY")

# Get API key for colab
# from google.colab import userdata

# API_KEY = userdata.get("OPENPAGERANK_API_KEY")

# Get API key for local
# from dotenv import load_dotenv
# import os

# load_dotenv()

# API_KEY = os.getenv("OPENPAGERANK_API_KEY")

if not API_KEY:
    raise ValueError("API key tidak ditemukan")

def extract_domain(url):
    """
    Get domain from URL
    example:
    https://www.example.com/news → example.com
    """
    try:
        parsed = urlparse(url)
        domain = parsed.netloc.lower()

        # remove www
        if domain.startswith("www."):
            domain = domain[4:]

        return domain
    except:
        return None


def get_openpagerank(domains):
    """
    Get Open PageRank value from list of domain (max 100 per request)
    """
    url = "https://openpagerank.com/api/v1.0/getPageRank"

    headers = {
        "API-OPR": API_KEY
    }

    # Format parameter make it suitable for the API
    params = [("domains[]", d) for d in domains]

    try:
        response = requests.get(
            url,
            headers=headers,
            params=params,
            timeout=10
        )

        # Handle error response
        if response.status_code == 401:
            print("Unauthorized: API key is wrong or not valid")
            return {}

        if response.status_code == 429:
            print("Rate limit, please wait for a while")
            return {}

        if response.status_code != 200:
            print("Error:", response.status_code)
            return {}

        data = response.json()

        result = {}
        for item in data.get("response", []):
            domain = item.get("domain")
            rank = item.get("page_rank_decimal")  # 0 - 10

            result[domain] = rank

        return result

    except requests.exceptions.Timeout:
        print("Request timeout")
        return {}

    except Exception as e:
        print("Exception:", e)
        return {}

In [ ]:
def is_valid_web_url(url):
    """
    URL filters that are suitable for pagerank
    """
    if not url:
        return False

    url = url.lower()

    # file
    if url.endswith((".pdf", ".csv", ".xls", ".xlsx", ".json")):
        return False

    # file storage / raw content (PDF, picture, dataset, etc)
    blocked_domains = [
        "storage.googleapis.com",
        "drive.google.com",
        "dropbox.com",
        "s3.amazonaws.com"
    ]

    return not any(d in url for d in blocked_domains)

In [ ]:
def scale_pagerank(pr_value, min_pr=0.0, max_pr=10.0):
    if max_pr == min_pr:
        return 0.0
    return (pr_value - min_pr) / (max_pr - min_pr)

In [ ]:
def safe_float(x):
    try:
        return float(x)
    except:
        return 0.0

In [ ]:
def source_authority(G_cs):
    """
    Compute and assign pagerank-based trust scores for all source nodes in the graph.

    Process:
      1. Collect all crawlable URLs (reference URL, stated_in website, publisher website)
         from source nodes
      2. Batch fetch PageRank for all unique domains via OpenPageRank API
      3. Assign pagerank and trust score to each source node based on priority rules:
         - Has reference URL              → pagerank from reference URL domain
         - No URL, stated_in has website  → pagerank from stated_in website
         - No URL, publisher has website  → pagerank from publisher website
         - No URL, no website, has label  → fixed fallback score (0.4)
         - No URL and no stated_in        → score 0.0
      4. Apply penalties based on page_status
    """

    # ------------------------------------------------------------------
    # STEP 1: Collect all crawlable URLs from source nodes
    # ------------------------------------------------------------------
    url_to_domain = {}  # url → domain

    for node, data in G_cs.nodes(data=True):
        if data.get("type") != "source":
            continue

        # Collect all three URL candidates in one pass
        for candidate_url in [
            data.get("url"),
            data.get("stated_in_website"),
            data.get("publisher_website"),   # ← added
        ]:
            if candidate_url and candidate_url not in url_to_domain and is_valid_web_url(candidate_url):
                domain = extract_domain(candidate_url)
                if domain:
                    url_to_domain[candidate_url] = domain

    # ------------------------------------------------------------------
    # STEP 2: Batch fetch PageRank for all unique domains
    # ------------------------------------------------------------------
    all_domains     = list(set(url_to_domain.values()))
    domain_rank_map = {}  # domain → raw pagerank (0–10)

    for i in range(0, len(all_domains), 100):
        batch = all_domains[i:i + 100]
        print(f"🔍 Fetching PageRank batch {i // 100 + 1} ({len(batch)} domains)")
        result = get_openpagerank(batch)
        domain_rank_map.update(result)
        time.sleep(1)

    # ------------------------------------------------------------------
    # STEP 3: Assign scaled pagerank + trust score to each source node
    # ------------------------------------------------------------------
    for node, data in G_cs.nodes(data=True):
        if data.get("type") != "source":
            continue

        ref_url          = data.get("url")
        si_website       = data.get("stated_in_website")
        publisher_website = data.get("publisher_website")   # ← added
        si_label         = data.get("stated_in_label")
        publisher_label  = data.get("publisher_label")      # ← added for fallback check
        page_status      = data.get("source_scrape_status")

        # Determine raw pagerank based on source info priority
        if ref_url and ref_url in url_to_domain:
            # Case 1: Has reference URL
            domain    = url_to_domain[ref_url]
            raw_pr    = safe_float(domain_rank_map.get(domain, 0))
            scaled_pr = scale_pagerank(raw_pr)
            note      = "pagerank:ref_url"

        elif si_website and si_website in url_to_domain:
            # Case 2: No URL but stated_in has official website
            domain    = url_to_domain[si_website]
            raw_pr    = safe_float(domain_rank_map.get(domain, 0))
            scaled_pr = scale_pagerank(raw_pr)
            note      = "pagerank:stated_in_website"

        elif publisher_website and publisher_website in url_to_domain:
            # Case 3: No URL, stated_in has no website, but publisher has website
            # Slightly discounted vs stated_in because it is one hop removed
            domain    = url_to_domain[publisher_website]
            raw_pr    = safe_float(domain_rank_map.get(domain, 0))
            scaled_pr = scale_pagerank(raw_pr)
            note      = "pagerank:publisher_website"

        elif si_label or publisher_label:
            # Case 4: Some source identity exists but no web presence to measure
            domain    = None
            scaled_pr = 0.4   # fixed fallback, already in 0–1 range
            note      = "fallback:no_website"

        else:
            # Case 5: No URL and no stated_in / publisher at all
            domain    = None
            scaled_pr = 0.0
            note      = "no_source_info"

        # Store intermediate values
        data["domain"]   = domain
        data["pagerank"] = scaled_pr
        data["note"]     = note

        # Apply page_status penalties to get final trust score
        if page_status in ["404_not_found", "server_error", "invalid_goid_page"]:
            # Severe punishment: broken or invalid page
            score = -1.0
        elif page_status not in ["success", None]:
            # Mild penalty: unknown status, timeout, connection_error, crawl_error
            score = 0.2
        elif note in ("fallback:no_website", "no_source_info"):
            # Fixed score for non-web sources, no normalization needed
            score = scaled_pr
        else:
            score = round(scaled_pr, 4)

        data["source_trust_score"] = score

In [ ]:
def compute_source_trust(claim, G_cs, claim_node):
    """
    Aggregate trust scores from all source nodes connected to a claim node.
    Requires source_authority(G_cs) to have been called beforehand so that
    source_trust_score is already populated on each source node.
    """
    # Get all source nodes connected to this claim
    sources = [
        s for s in G_cs.successors(claim_node)
        if G_cs.nodes[s].get("type") == "source"
    ]
    if not sources:
        return 0.0

    valid_scores = []
    penalties    = []

    for src_node in sources:
        # Read pre-computed trust score from node attribute
        score = G_cs.nodes[src_node].get("source_trust_score")
        if score is None:
            continue
        if score >= 0:
            valid_scores.append(score)
        else:
            penalties.append(score)  # negative

    # ======================
    # BASE SCORE
    # ======================
    if valid_scores:
        base_score    = sum(valid_scores) / len(valid_scores)
        # Bonus for having multiple highly trusted sources
        trusted_count = sum(1 for s in valid_scores if s >= 0.85)
        bonus         = 0.05 * max(0, trusted_count - 1)
        base_score    = min(base_score + bonus, 1.0)
    else:
        base_score = 0.0

    # ======================
    # PENALTY
    # ======================
    if penalties:
        penalty_factor = sum(penalties) / len(penalties)
        base_score = base_score + penalty_factor * 0.7

    return max(min(base_score, 1.0), -1.0)

In [ ]:
# STEP 1: Compute pagerank + trust score for all source nodes (run once)
source_authority(G_cs)

# STEP 2: Aggregate trust score per claim node
for node, data in G_cs.nodes(data=True):
    if data.get("type") != "claim":
        continue
    trust_score = compute_source_trust(data.get("claim"), G_cs, node)
    G_cs.nodes[node]["source_trust"] = trust_score
    print(f"✅ CLAIM: {data.get('claim')} | Source Trust: {trust_score:.4f}")

# Calculate Trustworthiness

In [ ]:
def compute_composite_trust(G, claim_node):

    node_data = G.nodes[claim_node]

    # ======================
    # CLAIM FEATURES
    # ======================
    llm = float(node_data.get("trustworthinessLLM") or 0)
    sim = float(node_data.get("similarity") or 0)
    kg = float(node_data.get("kg_trust") or 0)

    # ======================
    # SOURCE FEATURES
    # ======================
    sources = [
        s for s in G.successors(claim_node)
        if G.nodes[s].get("type") == "source"
    ]

    if sources:
        deg = []
        src_trust = []

        for s in sources:
            s_data = G.nodes[s]

            deg.append(s_data.get("degree") or 0)
            src_trust.append(s_data.get("source_trust_score") or 0)

        # Average 
        deg = sum(deg) / len(deg)
        src = sum(src_trust) / len(src_trust)


    else:
        deg = bet = src = 0.0

    # ======================
    # FINAL SCORE
    # ======================
    final_score = (
        0.30 * llm +
        0.25 * src +
        0.15 * sim +
        0.15 * kg +
        0.15 * deg
    )

    return max(min(final_score, 1.0), 0.0)

In [ ]:
# Assign composite trust score to each claim node
for node, data in G_cs.nodes(data=True):

    if data.get("type") != "claim":
        continue

    composite = compute_composite_trust(G_cs, node)
    G_cs.nodes[node]["trustworthiness"] = composite

In [ ]:
def compute_equal_weight_trust(G, claim_node):

    node_data = G.nodes[claim_node]

    # ======================
    # CLAIM FEATURES
    # ======================
    llm = float(node_data.get("trustworthinessLLM") or 0)
    sim = float(node_data.get("similarity") or 0)
    kg = float(node_data.get("kg_trust") or 0)

    # ======================
    # SOURCE FEATURES
    # ======================
    sources = [
        s for s in G.successors(claim_node)
        if G.nodes[s].get("type") == "source"
    ]

    if sources:

        deg = []
        src_trust = []

        for s in sources:

            s_data = G.nodes[s]

            deg.append(s_data.get("degree") or 0)
            src_trust.append(s_data.get("source_trust_score") or 0)

        deg = sum(deg) / len(deg)
        src = sum(src_trust) / len(src_trust)

    else:
        deg = src = 0.0

    final_score = np.mean([
        llm,
        src,
        sim,
        kg,
        deg,
    ])

    return max(min(final_score, 1.0), 0.0)

In [ ]:
for node, data in G_cs.nodes(data=True):

    if data.get("type") != "claim":
        continue

    equal_weight_score = compute_equal_weight_trust(G_cs, node)

    G_cs.nodes[node]["trustworthinessEqualWeight"] = equal_weight_score

# Timeliness

In [ ]:
DECAY_CONFIG = {
    "very_fast": 0.01,     # age
    "fast": 0.005,         # position
    "medium": 0.002,       # default
    "slow": 0.0005,        # biography
    "static": 0.0          # no decay, science facts, history
}

In [ ]:
def timeliness_score(claim):

    # get decay category from LLM
    category = claim.get("temporal_category", "medium")

    lambda_decay = DECAY_CONFIG.get(category, 0.002)

    chosen_date = claim["latest_date"]

    if not chosen_date:
        return 0.0

    try:
        d = chosen_date if isinstance(chosen_date, datetime) else parser.parse(chosen_date)

        if d.tzinfo is None:
            d = d.replace(tzinfo=timezone.utc)

        age_days = max(0, (datetime.now(timezone.utc) - d).days)  

        return math.exp(-lambda_decay * age_days)

    except Exception as e:
        print("Date parse error:", chosen_date, e)
        return 0.0

In [ ]:
# Handle cdt
def clean_date(date_str):
    if not date_str:
        return None
    return re.sub(r"[A-Z]{3,4}$", "", date_str).strip()

In [ ]:
# Compute timeliness score for each claim node
for node, data in G_cs.nodes(data=True):
    if data.get("type") != "claim":
        continue

    latest_date = None

    # Reset the latest date for this claim
    G_cs.nodes[node]["latest_date"] = None

    sources = [
        s for s in G_cs.successors(node)
        if G_cs.nodes[s].get("type") == "source"
    ]

    # Compute timeliness without date information if no sources exist
    if not sources:
        score = timeliness_score(data)

        print("\n==============================")
        print("CLAIM:", data.get("claim"))
        print("==============================")
        print("TEMPORAL CATEGORY:", data.get("temporal_category"))
        print("FINAL DATE USED:", None)
        print("TIMELINESS:", round(score, 4))

        G_cs.nodes[node]["timeliness"] = score
        continue

    print("\n==============================")
    print("CLAIM:", data.get("claim"))
    print("==============================")

    for src_node in sources:
        src_data = G_cs.nodes[src_node]
        url = src_data.get("url")

        # Skip sources without a reference URL
        if not url:
            continue

        print("🌐 Crawling:", url)

        # Reuse cached crawl results when available
        if src_data.get("crawl_data"):
            crawl_data = src_data["crawl_data"]
        else:
            crawl_data = crawl_text(url)
            src_data["crawl_data"] = crawl_data

        date_value = None

        # Priority 1: dates provided directly by Wikidata qualifiers
        if src_data.get("published"):
            date_value = src_data["published"]
        elif src_data.get("retrieved"):
            date_value = src_data["retrieved"]

        # Priority 2: dates extracted from the source content
        else:
            if crawl_data:
                date_value = (
                    crawl_data.get("published_date")
                    or crawl_data.get("modified_date")
                    or crawl_data.get("meta_date")
                    or crawl_data.get("time_tag")
                    or crawl_data.get("pattern_date")
                    or crawl_data.get("last_modified_header")
                    or crawl_data.get("copyright_year")
                )

        # Keep the most recent date among all sources
        try:
            if date_value:
                clean = clean_date(date_value)
                parsed = parser.parse(clean)

                print("   📅 Candidate date:", parsed)

                if latest_date is None or parsed > latest_date:
                    latest_date = parsed

        except Exception as e:
            print("   ⚠️ Date parse error:", date_value, e)

    # Store the most recent date on the claim node
    if latest_date:
        G_cs.nodes[node]["latest_date"] = latest_date

    score = timeliness_score(G_cs.nodes[node])

    category = G_cs.nodes[node].get("temporal_category", "medium")
    decay_rate = DECAY_CONFIG.get(category, 0.002)

    print("\n📊 TIMELINESS DEBUG")
    print("TEMPORAL CATEGORY:", category)
    print("DECAY RATE:", decay_rate)
    print("FINAL DATE USED:", G_cs.nodes[node]["latest_date"])
    print("TIMELINESS:", round(score, 4))

    # Store the final timeliness score
    G_cs.nodes[node]["timeliness"] = score

# Labeling The Node to Help in Classification

In [ ]:
def classify_claim_node(node_data, trust_threshold=0.4, time_threshold=0.5):

    trustworthiness = float(node_data.get("trustworthiness", 0))
    timeliness_score = float(node_data.get("timeliness", 0))

    if trustworthiness >= trust_threshold and timeliness_score >= time_threshold:
        return "trustworthy_and_timely"

    elif trustworthiness >= trust_threshold:
        return "trustworthy"

    elif timeliness_score >= time_threshold:
        return "timely"

    else:
        return "untrusted"

In [ ]:
def classify_claim_node_equal_weight(node_data, trust_threshold=0.4, time_threshold=0.5):

    trustworthinessEqualWeight = float(
        node_data.get("trustworthinessEqualWeight", 0)
    )
    timeliness_score = float(
        node_data.get("timeliness", 0)
    )

    if trustworthinessEqualWeight >= trust_threshold and timeliness_score >= time_threshold:
        return "trustworthy_and_timely"

    elif trustworthinessEqualWeight >= trust_threshold:
        return "trustworthy"

    elif timeliness_score >= time_threshold:
        return "timely"

    else:
        return "untrusted"

In [ ]:
# Assign label to each claim node based on composite trust (different weight) and timeliness
for node, data in G_cs.nodes(data=True):

    if data.get("type") != "claim":
        continue

    label = classify_claim_node(data)
    G_cs.nodes[node]["label"] = label

In [ ]:
# Print top claims by composite trust score
sorted_nodes = sorted(
    [
        (n, d) for n, d in G_cs.nodes(data=True)
        if d.get("type") == "claim"
    ],
    key=lambda x: x[1].get("trustworthiness", 0), # Sorted by composite trust score
    reverse=True # Descending order
)

print("\n🏆 Top Trusted Claims (Different Weight):\n")

for rank, (node_id, data) in enumerate(sorted_nodes[:100], start=1):

    claim_text = data.get("claim", "N/A")

    print(f"{rank:02d}. {claim_text}")
    print(f"    Trust={data.get('trustworthiness', 0):.3f}")
    print(f"    Time={data.get('timeliness', 0):.3f}")
    print(f"    Label={data.get('label', 'N/A')}\n")

In [ ]:
print("\n Claims Rank 101+ (Different Weight):\n")

for rank, (node_id, data) in enumerate(sorted_nodes[100:], start=101):

    claim_text = data.get("claim", "N/A")

    print(f"{rank:03d}. {claim_text}")
    print(f"    Trust={data.get('trustworthiness', 0):.3f}")
    print(f"    Time={data.get('timeliness', 0):.3f}")
    print(f"    Label={data.get('label', 'N/A')}\n")

In [ ]:
# After timeliness and trustworthiness values are set
visualize_graph_interactive(G_cs)

In [ ]:
# Assign label to each claim node based on composite trust (equal weight) and timeliness
for node, data in G_cs.nodes(data=True):

    if data.get("type") != "claim":
        continue

    label = classify_claim_node(data)
    G_cs.nodes[node]["label"] = label
for node, data in G_cs.nodes(data=True):

    if data.get("type") != "claim":
        continue

    label = classify_claim_node_equal_weight(data)

    G_cs.nodes[node]["labelEqualWeight"] = label

In [ ]:
sorted_nodes_equal_weight = sorted(
    [
        (n, d) for n, d in G_cs.nodes(data=True)
        if d.get("type") == "claim"
    ],
    key=lambda x: x[1].get("trustworthinessEqualWeight", 0),
    reverse=True
)

print("\n🏆 Top Trusted Claims (Equal Weight):\n")

for rank, (node_id, data) in enumerate(sorted_nodes[:100], start=1):

    claim_text = data.get("claim", "N/A")

    print(f"{rank:02d}. {claim_text}")
    print(f"    Trust={data.get('trustworthinessEqualWeight', 0):.3f}")
    print(f"    Time={data.get('timeliness', 0):.3f}")
    print(f"    Label={data.get('labelEqualWeight', 'N/A')}\n")

In [ ]:
print("\n Claims Rank 101+ (Equal Weight):\n")

for rank, (node_id, data) in enumerate(sorted_nodes_equal_weight[100:], start=101):

    claim_text = data.get("claim", "N/A")

    print(f"{rank:03d}. {claim_text}")
    print(f"    Trust={data.get('trustworthinessEqualWeight', 0):.3f}")
    print(f"    Time={data.get('timeliness', 0):.3f}")
    print(f"    Label={data.get('labelEqualWeight', 'N/A')}\n")

# Print all features value

In [ ]:
def print_claim_features(G, claim_node):

    node_data = G.nodes[claim_node]

    sources = [
        s for s in G.successors(claim_node)
        if G.nodes[s].get("type") == "source"
    ]

    similarity = float(node_data.get("similarity") or 0)
    timeliness = float(node_data.get("timeliness") or 0)
    llm        = float(node_data.get("trustworthinessLLM") or 0)
    kg         = float(node_data.get("kg_trust") or 0)

    degrees = []
    source_trusts = []

    for s in sources:

        s_data = G.nodes[s]

        degrees.append(
            float(s_data.get("degree") or 0)
        )

        source_trusts.append(
            float(s_data.get("source_trust_score") or 0)
        )

    degree = (
        sum(degrees) / len(degrees)
        if degrees else 0.0
    )

    src = (
        sum(source_trusts) / len(source_trusts)
        if source_trusts else 0.0
    )

    print(f"    Similarity         : {similarity:.3f}")
    print(f"    Timeliness         : {timeliness:.3f}")
    print(f"    LLM Trust Score    : {llm:.3f}")
    print(f"    KG Trust Score     : {kg:.3f}")
    print(f"    Source Trust Score : {src:.3f}")
    print(f"    Degree Centrality  : {degree:.6f}")

In [ ]:
# ----------------------------------------- Print all metrics ------------------------------------------------
sorted_nodes = sorted(
    [
        (n, d)
        for n, d in G_cs.nodes(data=True)
        if d.get("type") == "claim"
    ],
    key=lambda x: x[1].get("trustworthiness", 0),
    reverse=True
)

print("\n Claims features (Different Weight)\n")

for rank, (node_id, data) in enumerate(sorted_nodes, start=1):

    print(f"{rank:02d}. {data.get('claim', 'N/A')}")
    print(f"    Trust={data.get('trustworthiness', 0):.3f}")
    print(f"    Time={data.get('timeliness', 0):.3f}")
    print(f"    Label={data.get('label', 'N/A')}")

    print_claim_features(G_cs, node_id)

    print()

In [ ]:
# ----------------------------------------- Print all metrics------------------------------------------------
sorted_nodes_equal = sorted(
    [
        (n, d)
        for n, d in G_cs.nodes(data=True)
        if d.get("type") == "claim"
    ],
    key=lambda x: x[1].get("trustworthinessEqualWeight", 0),
    reverse=True
)

print("\n Claims features (Equal Weight)\n")

for rank, (node_id, data) in enumerate(sorted_nodes_equal, start=1):

    print(f"{rank:02d}. {data.get('claim', 'N/A')}")
    print(f"    Trust={data.get('trustworthinessEqualWeight', 0):.3f}")
    print(f"    Time={data.get('timeliness', 0):.3f}")
    print(f"    Label={data.get('labelEqualWeight', 'N/A')}")

    print_claim_features(G_cs, node_id)

    print()

# Summary

In [ ]:
# ======================
# SUMMARY (GRAPH BASED)
# ======================

claim_nodes = [
    n for n, d in G_cs.nodes(data=True)
    if d.get("type") == "claim"
]

total_claims = len(claim_nodes)

with_ref = 0
no_ref = 0

for c in claim_nodes:

    sources = [
        s for s in G_cs.successors(c)
        if G_cs.nodes[s].get("type") == "source"
    ]

    if sources:
        with_ref += 1   
    else:
        no_ref += 1

print("📌 FINAL SUMMARY")
print(f"  - Total claims      : {total_claims}")
print(f"  - With reference    : {with_ref}")
print(f"  - Without reference : {no_ref}")